# Diagnóstico del Modelo 3 — Notebook orquestador

## 1. Configuración

In [ ]:
import os, sys

# --- RUTAS 
# Carpeta que contiene este notebook y el paquete diagnosticos_m3/
CARPETA_PIPELINE = r"C:/Users/USUARIO/OneDrive - Universidad Nacional de Colombia/Tesis/Escritorio/Códigos/Modelos/Modelo 3/3_Diagnosticos_Python"

# Carpeta con las muestras MCMC
DIR_MUESTRAS = r"C:/Users/USUARIO/OneDrive - Universidad Nacional de Colombia/Tesis/Escritorio/Códigos/Modelos/Modelo 3/mcmc_salida_m3"

# Carpeta de salida de figuras/tablas/reporte
DIR_SALIDA = r"C:/Users/USUARIO/OneDrive - Universidad Nacional de Colombia/Tesis/Escritorio/Códigos/Modelos/Modelo 3/1_Resultados_diagnosticos_python"

# --- OPCIONES ---
MODO_RAPIDO   = False   # True = pasada rápida de validación (submuestreo)
SIN_LOO       = False   # True = omite PSIS-LOO (costoso en N grande); usa WAIC
N_CARGAR      = 6000    # muestras posteriores a cargar (rápido lo baja a 400)
N_REP_PPC     = 200     # réplicas para PPC (rápido lo baja a 40)
SEED          = 0

# --- registrar el paquete ---
if CARPETA_PIPELINE not in sys.path:
    sys.path.insert(0, CARPETA_PIPELINE)

print("Configuración lista.")
print("  Muestras :", DIR_MUESTRAS)
print("  Salida   :", DIR_SALIDA)
print("  Modo     :", "RÁPIDO" if MODO_RAPIDO else "COMPLETO",
      "| sin-LOO" if SIN_LOO else "| con LOO")

Configuración lista.
  Muestras : C:/Users/USUARIO/OneDrive - Universidad Nacional de Colombia/Tesis/Escritorio/Códigos/Modelos/Modelo 3/1_Resultados_mcmc_output
  Salida   : C:/Users/USUARIO/OneDrive - Universidad Nacional de Colombia/Tesis/Escritorio/Códigos/Modelos/Modelo 3/2_Diagnosticos_python
  Modo     : COMPLETO | con LOO


## 2. Preparar metadatos y cargar muestras

In [2]:
import time, warnings
import numpy as np
from scipy import stats

from diagnosticos_m3 import (
    io_muestras as io, convergencia as cv, dinamico as din,
    tau2_diag as t2, predictivo as pr, figuras as fig, reporte as rep,
)
from diagnosticos_m3 import preparar_metadatos as pm

os.makedirs(DIR_SALIDA, exist_ok=True)
_t_ini = time.time()

# metadatos.json (autogenera desde .rds si falta)
pm.asegurar_metadatos_json(DIR_MUESTRAS, verbose=True)

# ajustes de modo
_n_cargar = min(N_CARGAR, 400) if MODO_RAPIDO else N_CARGAR
_n_rep_ppc = min(N_REP_PPC, 40) if MODO_RAPIDO else N_REP_PPC
_submuestreo_ll = 4 if MODO_RAPIDO else 1
if MODO_RAPIDO:
    print(">> MODO RAPIDO: submuestreo de cadenas y observaciones.")

# cargar muestras
muestras = io.cargar_todo(DIR_MUESTRAS, n_cargar=_n_cargar, verbose=True)
meta = muestras.meta

# datos observados (para las secciones predictivas)
datos = io.cargar_datos_observados(DIR_MUESTRAS, meta)
hay_datos = datos is not None
if hay_datos:
    print("[OK] Datos observados: N={}, p={}".format(datos.N_total, datos.p))
else:
    print("[!] Sin datos_export/: se omitiran las secciones predictivas.")

# contexto compartido para el reporte final
ctx = {"meta": meta, "n_save": muestras.n_save, "modo_rapido": MODO_RAPIDO}
S = muestras.n_save
idx_ll = np.arange(0, S, _submuestreo_ll)

✓ metadatos.json ya existe y es válido (metadatos.json)
Cargando muestras 1 a 6000 (6000 muestras) desde 1_Resultados_mcmc_output/
RAM usada por muestras: 354 MB
  theta (83, 17, 6000)  s (33, 16, 6000)  v (125, 33, 6000)
  tau2_E (63, 16, 6000)  tau2_M (11, 16, 6000)  tau2_D (8, 16, 6000)
[OK] Datos observados: N=499983, p=83


In [3]:
import sys
print(sys.version)

3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]


## 3. Análisis dinámico

In [7]:
print("Evolución del intercepto...")
evol_mu = din.evolucion_intercepto(muestras)
fig.fig_evolucion_intercepto(muestras, evol_mu, DIR_SALIDA)

innov_all, shrink_all = {}, {}
for B in ("E", "M", "D"):
    print(f"Bloque {B}: coeficientes + LASSO temporal...")
    evol = din.evolucion_bloque(muestras, B)
    top = din.top_dinamicos(evol, n=9)
    fig.fig_evolucion_bloque(muestras, B, evol, top, DIR_SALIDA)

    innov = din.innovaciones_bloque(muestras, B)
    innov_all[B] = innov
    fig.fig_lasso_actividad(muestras, B, innov, DIR_SALIDA)
    q_th, q_emp, _ = din.qq_laplace(innov["omega_pool"])
    fig.fig_lasso_qq(B, q_th, q_emp, DIR_SALIDA)

    shrink_all[B] = din.shrinkage_medio_por_periodo(muestras, B)

fig.fig_shrinkage_temporal(muestras, DIR_SALIDA, shrink_all)

print("Varianza departamental s_kt...")
dvar = din.dinamica_varianza(muestras)
fig.fig_varianza_departamental(muestras, dvar, DIR_SALIDA)

print("Tabla resumen de parámetros...")
resumen_full = din.tabla_resumen(muestras)
rep.escribir_csv(resumen_full, os.path.join(DIR_SALIDA, "resumen_theta_full.csv"))
resumen_T = [f for f in resumen_full if f["t_idx"] == meta.Tt]
rep.escribir_csv(resumen_T, os.path.join(DIR_SALIDA, "resumen_theta_T.csv"))

ctx.update({"innov": innov_all, "dvar": dvar, "resumen_T": resumen_T})
print("✓ Sección dinámica completada.")

Evolución del intercepto...
Bloque E: coeficientes + LASSO temporal...
Bloque M: coeficientes + LASSO temporal...
Bloque D: coeficientes + LASSO temporal...
Varianza departamental s_kt...
Tabla resumen de parámetros...
✓ Sección dinámica completada.


## 4. Convergencia MCMC

In [6]:
def diagnosticar_convergencia(muestras, dir_out):
    meta = muestras.meta
    Tt = meta.Tt
    periodos = cv.periodos_diagnostico(Tt)

    mu_chain = muestras.theta[0, :, :].T
    ess_mu = cv.ess_por_columnas(mu_chain, usar_arviz=True)
    gew_mu = np.array([cv.geweke_z(mu_chain[:, j]) for j in range(mu_chain.shape[1])])
    rhat_mu = np.array([cv.split_rhat(mu_chain[:, j]) for j in range(mu_chain.shape[1])])

    ess_lambda = cv.ess_por_columnas(muestras.lambda2, usar_arviz=True)
    rhat_lambda = np.array([cv.split_rhat(muestras.lambda2[:, j]) for j in range(3)])

    ess_theta, rhat_theta, gew_theta_list = {}, {}, []
    for t in periodos:
        chain_t = muestras.theta[:, t, :].T
        ess_theta[f"t{t}"] = cv.ess_por_columnas(chain_t, usar_arviz=True)
        rhat_theta[f"t{t}"] = np.array([cv.split_rhat(chain_t[:, j])
                                        for j in range(chain_t.shape[1])])
        gew_theta_list.append(np.array([cv.geweke_z(chain_t[:, j])
                                        for j in range(chain_t.shape[1])]))

    s = muestras.s
    m_, Ttt, _ = s.shape
    ess_s = np.full(m_ * Ttt, np.nan)
    idx = 0
    for kk in range(m_):
        for tt in range(Ttt):
            x = s[kk, tt, :]
            if np.unique(x).size >= 8:
                ess_s[idx] = cv.ess_geyer(x)
            idx += 1

    gew_all = np.concatenate([gew_mu,
                              np.array([cv.geweke_z(muestras.lambda2[:, j])
                                        for j in range(3)])] + gew_theta_list)
    gew_pfdr = cv.geweke_fdr(gew_all)
    hw = [cv.heidel_welch(mu_chain[:, j]) for j in range(mu_chain.shape[1])]

    print(f"  ESS μ_t: min={np.nanmin(ess_mu):.0f}, mediana={np.nanmedian(ess_mu):.0f}")
    print(f"  ESS λ²: {', '.join(f'{v:.0f}' for v in ess_lambda)}")
    theta_all = np.concatenate([np.asarray(v).ravel() for v in ess_theta.values()])
    print(f"  ESS θ (multi-t): min={np.nanmin(theta_all):.0f}, "
          f"mediana={np.nanmedian(theta_all):.0f}")
    print(f"  ESS s_kt: min={np.nanmin(ess_s):.0f}, mediana={np.nanmedian(ess_s):.0f}")
    rhat_all = np.concatenate([rhat_mu, rhat_lambda,
                               np.concatenate(list(rhat_theta.values()))])
    print(f"  R̂ máximo (split rank-norm): {np.nanmax(rhat_all):.3f}")
    print(f"  Heidelberger-Welch (bm): {sum(1 for h in hw if h['stest']==0)}/{len(hw)} "
          f"μ_t fallan estacionariedad")
    print(f"  Geweke tras FDR: {int(np.sum(gew_pfdr<0.05))}/"
          f"{int(np.sum(np.isfinite(gew_pfdr)))} significativos")

    return cv.ResumenConvergencia(
        ess_mu=ess_mu, ess_lambda=ess_lambda, ess_theta=ess_theta,
        ess_s=ess_s, geweke_z=gew_all, geweke_pfdr=gew_pfdr,
        rhat_mu=rhat_mu, rhat_lambda=rhat_lambda, rhat_theta=rhat_theta,
        heidel_mu=hw, raftery={}, periodos_diag=periodos)

diag_conv = diagnosticar_convergencia(muestras, DIR_SALIDA)
ctx["diag_conv"] = diag_conv

rl = cv.raftery_lewis(muestras.lambda2[:, 0])
ctx["raftery"] = rl
print(f"  Raftery-Lewis λ²_E: N_min={rl['N_min']}, N={rl['N']}, I={rl['I']:.2f}")

fig.fig_traceplots(muestras, DIR_SALIDA)
fig.fig_ess_resumen(diag_conv, DIR_SALIDA)
print("✓ Convergencia completada.")

  ESS μ_t: min=4866, mediana=5920
  ESS λ²: 944, 920, 2017
  ESS θ (multi-t): min=4055, mediana=6000
  ESS s_kt: min=101, mediana=649
  R̂ máximo (split rank-norm): 1.003
  Heidelberger-Welch (bm): 0/17 μ_t fallan estacionariedad
  Geweke tras FDR: 0/435 significativos
  Raftery-Lewis λ²_E: N_min=3746, N=8295, I=2.22
✓ Convergencia completada.


## 5. Diagnóstico de τ² (LASSO dinámico)

In [6]:
tau2_all, bimod_all = {}, {}
for B in ("E", "M", "D"):
    print(f"Bloque {B}...")
    d = t2.diagnosticar_bloque(muestras, B)
    tau2_all[B] = d
    fig.fig_tau2_ess_heatmap(d, DIR_SALIDA)
    fig.fig_tau2_magnitud(d, DIR_SALIDA)
    if not MODO_RAPIDO:
        bimod_all[B] = t2.detectar_bimodalidad(muestras, B)
    print(f"  ESS(log τ²): min={d['ess_min']:.0f}, mediana={d['ess_med']:.0f}, "
          f"max={d['ess_max']:.0f} | ESS<200: {d['n_low_ess']}")

comp = t2.comparar_bloques(muestras)
fig.fig_tau2_comparacion(comp, DIR_SALIDA)
acop = t2.acoplamiento_omega_tau(muestras, tau2_all)
fig.fig_tau2_acoplamiento(acop, DIR_SALIDA)
san = t2.sanity_lambda_tau(muestras)
for B in ("E", "M", "D"):
    s_ = san[B]
    print(f"  {'✓' if s_['ok'] else '⚠'} λ²↔τ² {B}: E[τ²|λ̂²]={s_['esperado']:.4f} "
          f"vs mediana(τ²)={s_['observado']:.4f} (ratio={s_['ratio']:.2f})")
ctx.update({"tau2": tau2_all, "bimod": bimod_all})

fig.fig_lambda2_comparacion(muestras, DIR_SALIDA)
print("✓ Sección τ² completada.")

Bloque E...
  ESS(log τ²): min=3589, mediana=5735, max=6000 | ESS<200: 0
Bloque M...
  ESS(log τ²): min=2916, mediana=4871, max=5826 | ESS<200: 0
Bloque D...
  ESS(log τ²): min=4482, mediana=5454, max=6000 | ESS<200: 0
  ✓ λ²↔τ² E: E[τ²|λ̂²]=17.6685 vs mediana(τ²)=11.4113 (ratio=0.65)
  ✓ λ²↔τ² M: E[τ²|λ̂²]=3.9757 vs mediana(τ²)=2.7195 (ratio=0.68)
  ✓ λ²↔τ² D: E[τ²|λ̂²]=31.6068 vs mediana(τ²)=22.4802 (ratio=0.71)
✓ Sección τ² completada.


## 6. Diagnóstico predictivo

In [4]:
if hay_datos:
    # Submuestrear a máx 1200 muestras SOLO para el predictivo (evita
    # la matriz log_lik de 22 GB; el elpd converge mucho antes de 6000).
    N_MAX_PRED = 1200
    paso = max(1, S // N_MAX_PRED)
    idx_ll = np.arange(0, S, paso)
    print(f"Predictivo usa {len(idx_ll)} de {S} muestras "
          f"(matriz log_lik ~{len(idx_ll)*datos.N_total*4/1e9:.1f} GB)")

    print(f"log-verosimilitud puntual ({len(idx_ll)} muestras × {datos.N_total} obs)...")
    log_lik = pr.log_lik_pointwise(datos, muestras, idx_muestras=idx_ll, verbose=True)

    waic_res = pr.waic(log_lik)
    ctx["waic"] = waic_res
    print(f"  WAIC: elpd={waic_res['elpd_waic']:.2f} (±{waic_res['se']:.2f}), "
          f"p_eff={waic_res['p_waic']:.2f}")

    loo_res = None
    if not SIN_LOO:
        print("PSIS-LOO (motor ArviZ)...")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            loo_res = pr.psis_loo(log_lik, usar_arviz=True)
        print(f"  LOO: elpd={loo_res['elpd_loo']:.2f} (±{loo_res['se']:.2f}), "
              f"p_eff={loo_res['p_loo']:.2f}, k>0.7: {loo_res['n_k_altos']} "
              f"[{loo_res['motor']}]")
    else:
        print("  (PSIS-LOO omitido por SIN_LOO)")
    ctx["loo"] = loo_res
    rep.tabla_comparacion_modelo_csv(waic_res, loo_res, DIR_SALIDA)
    del log_lik

    print("PIT exacto de la mezcla predictiva...")
    pit_vals = pr.pit_mezcla(datos, muestras, idx_muestras=idx_ll, verbose=True)
    ks_p = float(stats.kstest(pit_vals, "uniform").pvalue)
    fig.fig_pit(pit_vals, datos.k_all, DIR_SALIDA, ks_p=ks_p)
    ctx["pit"] = {"ks_p": ks_p}
    print(f"  KS PIT vs U(0,1): p={ks_p:.4f}")

    print(f"PPC de estadísticos ({_n_rep_ppc} réplicas)...")
    ppc = pr.ppc_estadisticos(datos, muestras, n_rep=_n_rep_ppc, seed=SEED, verbose=True)
    fig.fig_ppc_estadisticos(ppc, DIR_SALIDA)
    fig.fig_ppc_temporal(muestras, ppc, DIR_SALIDA)
    ctx["ppc"] = ppc

    print("fair-CRPS por estrato...")
    crps = pr.fair_crps_normal(datos, muestras, idx_muestras=idx_ll, verbose=True)
    fig.fig_crps_heatmap(crps, datos.k_all, datos.t_all, meta, DIR_SALIDA)
    print(f"  CRPS promedio: {crps.mean():.3f}")

    print("Residuos estructurales...")
    residuos = pr.residuos_estructurales(datos, muestras)
    fig.fig_residuos(residuos, datos.k_all, datos.t_all, DIR_SALIDA, seed=SEED)
    ctx["residuos"] = residuos
    print(f"  RMSE={residuos['rmse']:.3f}  MAE={residuos['mae']:.3f}")
    print("✓ Sección predictiva completada.")
else:
    print("⚠ Sección predictiva OMITIDA (sin datos observados).")

Predictivo usa 1200 de 6000 muestras (matriz log_lik ~2.4 GB)
log-verosimilitud puntual (1200 muestras × 499983 obs)...
    log_lik: 120/1200 muestras
    log_lik: 240/1200 muestras
    log_lik: 360/1200 muestras
    log_lik: 480/1200 muestras
    log_lik: 600/1200 muestras
    log_lik: 720/1200 muestras
    log_lik: 840/1200 muestras
    log_lik: 960/1200 muestras
    log_lik: 1080/1200 muestras
    log_lik: 1200/1200 muestras
  WAIC: elpd=-2599451.28 (±497.31), p_eff=1945.85
PSIS-LOO (motor ArviZ)...
  LOO: elpd=-2599462.27 (±497.36), p_eff=1956.84, k>0.7: 6 [propio]
PIT exacto de la mezcla predictiva...
    PIT: 120/1200 muestras
    PIT: 240/1200 muestras
    PIT: 360/1200 muestras
    PIT: 480/1200 muestras
    PIT: 600/1200 muestras
    PIT: 720/1200 muestras
    PIT: 840/1200 muestras
    PIT: 960/1200 muestras
    PIT: 1080/1200 muestras
    PIT: 1200/1200 muestras
  KS PIT vs U(0,1): p=0.0000
PPC de estadísticos (200 réplicas)...
    PPC y_rep: 40/200
    PPC y_rep: 80/200
   

c:\Users\USUARIO\anaconda3\Lib\site-packages\numpy\core\_methods.py:118: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


  RMSE=43.891  MAE=35.317
✓ Sección predictiva completada.


## 7. Tablas y reporte consolidado

In [8]:
rep.tabla_convergencia_csv(diag_conv, DIR_SALIDA)
ruta_md = rep.generar_reporte_md(ctx, DIR_SALIDA)

n_pdf = len([f for f in os.listdir(DIR_SALIDA) if f.endswith(".pdf")])
n_csv = len([f for f in os.listdir(DIR_SALIDA) if f.endswith(".csv")])
print(f"\n✓ Diagnóstico completado en {time.time()-_t_ini:.1f}s")
print(f"  Figuras PDF : {n_pdf}")
print(f"  Tablas CSV  : {n_csv}")
print(f"  Reporte     : {os.path.basename(ruta_md)}")
print(f"  Directorio  : {DIR_SALIDA}")


✓ Diagnóstico completado en 1299.4s
  Figuras PDF : 29
  Tablas CSV  : 4
  Reporte     : reporte_diagnostico_M3.md
  Directorio  : C:/Users/USUARIO/OneDrive - Universidad Nacional de Colombia/Tesis/Escritorio/Códigos/Modelos/Modelo 3/2_Diagnosticos_python
